In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/simpleserial-xor
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
Compiling:
Compiling:
.
Compiling:
Compiling:
-en     simpleserial-xor.c ...
-en     .././simpleserial/simpleserial.c ...
Compiling:
-en     .././hal/hal.c ...
-en     .././hal//sam4s/startup_sam4s.c ...
-en     .././hal//sam4s/sam4s_hal.c ...
.
.
Compiling:
Compiling:
-en     .././hal//sam4s/uart.c ...
-en     .././hal//sam4s/pio.c ...
.
Compiling:
.
.
-en     .././hal//sam4s/system_sam4s.c ...
Compiling:
Compiling:
-en     .././hal//sam4s/sysclk.c ...
-en     .././hal//sam4s/pmc.c ...
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     simp

In [4]:
import subprocess
from IPython.display import HTML, display
import html

cmd = f"""
cd ../../firmware/mcu/simpleserial-xor
make PLATFORM={PLATFORM} CRYPTO_TARGET=NONE SS_VER={SS_VER} -j
"""

result = subprocess.run( cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

display(HTML(f"""
<div style="max-height: 500px; overflow-y: auto; border: 1px solid #888; padding: 8px; background-color: #111; color: #ddd; font-family: monospace;
    white-space: pre-wrap;">
{html.escape(result.stdout)}
</div>
"""))

In [5]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-xor/simpleserial-xor-{}.hex".format(PLATFORM))

In [6]:

print(target)

SimpleSerial2 Settings =
	output_len     = 16
	baud           = 230400
	xonxoff        = None
	currently_xoff = None
	parity         = none
	stop_bits      = 1
	


list all avaibable commands

In [7]:
# One useful correction to the w test
target.flush()

target.simpleserial_write('w', bytearray())

cmds = target.simpleserial_read('r', 4, ack=False)

print("Raw:", cmds)
print("Commands:", [chr(x) for x in cmds])

Raw: CWbytearray(b'76 77 61 62')
Commands: ['v', 'w', 'a', 'b']


write only the key

In [8]:
# The next issue is probably the V2 ACK/status packet sequencing, rather than an unknown command.

ktp = cw.ktp.Basic()
key, text = ktp.next()

print("key :", key)
print("text:", text)

target.flush()

# Set key
target.simpleserial_write('b', key)

# Explicitly consume the V2 acknowledgement
ack = target.simpleserial_wait_ack()

print("b ACK:", ack)

key : CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')
text: CWbytearray(b'f5 af 0b 72 35 8b b3 cf 0c f5 a8 21 92 22 86 a5')
b ACK: bytearray(b'\x00')


write the plaintext

In [9]:
def hexstr(data):
    return " ".join(f"{b:02X}" for b in data)

target.simpleserial_write('a', text)

response = target.simpleserial_read('r', 16)

expected = bytearray(t ^ k for t, k in zip(text, key))

print("received:", hexstr(response))
print("expected:", hexstr(expected))
print("match   :", response == expected)

received: DE D1 1E 64 1D 25 61 69 A7 02 BD A9 9B ED C9 99
expected: DE D1 1E 64 1D 25 61 69 A7 02 BD A9 9B ED C9 99
match   : True


In [10]:
# Then test only b:
ktp = cw.ktp.Basic()
key, text = ktp.next()

print("key:", hexstr(key))

target.simpleserial_write('b', key)

ack = target.simpleserial_wait_ack()
print("b ack:", hexstr(ack))

print("remaining:", target.in_waiting())

key: 2B 7E 15 16 28 AE D2 A6 AB F7 15 88 09 CF 4F 3C
b ack: 00
remaining: 0


In [11]:
# test a:
target.simpleserial_write('a', text)

response = target.simpleserial_read('r', 16)

print("response:", hexstr(response))

expected = bytearray(t ^ k for t, k in zip(text, key))

print("expected:", hexstr(expected))
print("match:", response == expected)

response: F0 B2 AA 19 1B 72 88 0F 67 FB A5 A4 56 81 CE 8B
expected: F0 B2 AA 19 1B 72 88 0F 67 FB A5 A4 56 81 CE 8B
match: True


In [12]:
ktp = cw.ktp.Basic()
key, text = ktp.next()

target.flush()

# Set key
target.simpleserial_write('b', key)

ack = target.simpleserial_wait_ack()

if ack != bytearray([0]):
    raise RuntimeError(f"Set-key failed: {ack}")

# XOR
target.simpleserial_write('a', text)
response = target.simpleserial_read('r', 16)

expected = bytearray(t ^ k for t, k in zip(text, key))
print("key     :", hexstr(key))
print("text    :", hexstr(text))
print("response:", hexstr(response))
print("expected:", hexstr(expected))

if response is not None:
    print("match   :", bytes(response) == bytes(expected))

key     : 2B 7E 15 16 28 AE D2 A6 AB F7 15 88 09 CF 4F 3C
text    : FA B0 62 03 84 32 89 13 3D 1A 7D CF 38 FC 04 C1
response: D1 CE 77 15 AC 9C 5B B5 96 ED 68 47 31 33 4B FD
expected: D1 CE 77 15 AC 9C 5B B5 96 ED 68 47 31 33 4B FD
match   : True


In [13]:
ktp = cw.ktp.Basic()
key, text = ktp.next()

target.flush()

target.simpleserial_write('b', key)
ack = target.simpleserial_wait_ack()

target.simpleserial_write('a', text)

print("waiting:", target.in_waiting())
print("read:", repr(target.read(timeout=1000)))
#response = target.simpleserial_read('r', 16)
#print(response)

#expected = bytearray(t ^ k for t, k in zip(text, key))

#print("received:", response)
#print("expected:", expected)
#print("match   :", response == expected)

waiting: 0
read: '\x14r\x10÷;þ\x8b\x9bB'
